In [2]:
import requests
import pandas as pd
from pathlib import Path

BASE_DIR   = Path(r"D:\Shikha\bias_project")
SSA_DIR    = BASE_DIR / "validation_data" / "ssa_names"
OUTPUT_DIR = BASE_DIR / "word_lists" / "era_names"
SSA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# SSA's own server hard-blocks programmatic access (403 even with UA).
# Mirror: hackerb9/ssa-baby-names hosts unzipped yobYYYY.txt in native SSA format.
RAW_BASES = [
    "https://raw.githubusercontent.com/hackerb9/ssa-baby-names/master/raw-data",
    "https://raw.githubusercontent.com/hackerb9/ssa-baby-names/main/raw-data",
]

# Birth-cohort windows matching our corpus periods (adults, not babies-of-the-decade)
ERA_WINDOWS = {
    "historical_1920s": range(1870, 1901),   # adults in 1920-1922 text
    "modern_2024":      range(1950, 2006),   # adults in 2020 Reddit / 2024 news+CR
}

needed_years = sorted({y for yrs in ERA_WINDOWS.values() for y in yrs})

def fetch_yob(year):
    """Download one yobYYYY.txt from the mirror; return text or None."""
    local = SSA_DIR / f"yob{year}.txt"
    if local.exists() and local.stat().st_size > 100:
        return local.read_text(encoding="utf-8")
    for base in RAW_BASES:
        try:
            r = requests.get(f"{base}/yob{year}.txt", timeout=30)
            if r.status_code == 200 and len(r.content) > 100:
                local.write_bytes(r.content)
                return r.text
        except Exception as e:
            print(f"  {year} via {base.split('/')[-2]}: {e}")
    return None

frames, missing = [], []
print(f"Fetching {len(needed_years)} year files from mirror ({needed_years[0]}–{needed_years[-1]})...")
for y in needed_years:
    txt = fetch_yob(y)
    if txt is None:
        missing.append(y)
        continue
    df = pd.read_csv(SSA_DIR / f"yob{y}.txt", header=None, names=["name", "sex", "count"])
    df["year"] = y
    frames.append(df)

ssa = pd.concat(frames, ignore_index=True)
ssa["name"] = ssa["name"].str.lower()
print(f"\nLoaded {len(frames)} year files, {len(ssa):,} name-year records")
print(f"Year range: {ssa['year'].min()}–{ssa['year'].max()}")
if missing:
    print(f"MISSING years (check branch/availability): {missing}")

Fetching 87 year files from mirror (1870–2005)...

Loaded 77 year files, 1,113,420 name-year records
Year range: 1880–2005
MISSING years (check branch/availability): [1870, 1871, 1872, 1873, 1874, 1875, 1876, 1877, 1878, 1879]


In [3]:
import requests
from pathlib import Path

SSA_DIR = Path(r"D:\Shikha\bias_project\validation_data\ssa_names")
SSA_DIR.mkdir(parents=True, exist_ok=True)
SSA_ZIP = SSA_DIR / "names.zip"

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

r = requests.get("https://www.ssa.gov/oact/babynames/names.zip", headers=HEADERS, timeout=60)
print(f"Status: {r.status_code}")
print(f"Content length: {len(r.content)} bytes")
if r.status_code == 200:
    SSA_ZIP.write_bytes(r.content)
    print(f"Saved. File exists: {SSA_ZIP.exists()}, size: {SSA_ZIP.stat().st_size} bytes")

Status: 403
Content length: 399 bytes


In [4]:
TOP_N        = 20      # names per sex per era
PURITY_MIN   = 0.95    # keep names ≥95% one sex (drop unisex)
MIN_TOTAL    = 100     # ignore very rare names (noise)

def build_era_lists(ssa, year_window, era_label):
    sub = ssa[ssa["year"].isin(year_window)]

    # Sum counts per name × sex across the window
    pivot = (sub.groupby(["name", "sex"])["count"].sum()
                .unstack(fill_value=0)
                .rename(columns={"M": "m", "F": "f"}))
    for col in ("m", "f"):
        if col not in pivot:
            pivot[col] = 0
    pivot["total"]  = pivot["m"] + pivot["f"]
    pivot["purity"] = pivot[["m", "f"]].max(axis=1) / pivot["total"]
    pivot = pivot[pivot["total"] >= MIN_TOTAL]

    male = (pivot[(pivot["m"] >= pivot["f"]) & (pivot["purity"] >= PURITY_MIN)]
                .sort_values("m", ascending=False).head(TOP_N).index.tolist())
    female = (pivot[(pivot["f"] > pivot["m"]) & (pivot["purity"] >= PURITY_MIN)]
                .sort_values("f", ascending=False).head(TOP_N).index.tolist())

    (OUTPUT_DIR / f"male_names_{era_label}.txt").write_text("\n".join(male) + "\n", encoding="utf-8")
    (OUTPUT_DIR / f"female_names_{era_label}.txt").write_text("\n".join(female) + "\n", encoding="utf-8")
    return male, female

print(f"Building era name lists (top {TOP_N}/sex, purity ≥{PURITY_MIN})\n")
for era_label, window in ERA_WINDOWS.items():
    yrs = [y for y in window if y >= 1880]
    male, female = build_era_lists(ssa, yrs, era_label)
    print(f"=== {era_label}  ({yrs[0]}–{yrs[-1]}) ===")
    print(f"  male  ({len(male)}): {', '.join(male[:10])} ...")
    print(f"  female({len(female)}): {', '.join(female[:10])} ...")
    print()

# Sanity check: how much do historical vs modern lists overlap?
import itertools
files = {f.stem: set(f.read_text().split()) for f in OUTPUT_DIR.glob("*.txt")}
print("Cross-era overlap (lower = more era-specific):")
for a, b in itertools.combinations(sorted(files), 2):
    if a.split("_")[0] == b.split("_")[0]:  # compare same sex across eras
        inter = files[a] & files[b]
        print(f"  {a} ∩ {b}: {len(inter)} shared — {', '.join(sorted(inter)) or '(none)'}")

Building era name lists (top 20/sex, purity ≥0.95)

=== historical_1920s  (1880–1900) ===
  male  (20): john, william, james, george, charles, frank, joseph, robert, henry, edward ...
  female(20): mary, anna, margaret, elizabeth, emma, helen, florence, ethel, minnie, bertha ...

=== modern_2024  (1950–2005) ===
  male  (20): michael, david, james, john, robert, william, christopher, joseph, daniel, richard ...
  female(20): jennifer, mary, jessica, lisa, elizabeth, linda, susan, patricia, kimberly, karen ...

Cross-era overlap (lower = more era-specific):
  female_names_historical_1920s ∩ female_names_modern_2024: 2 shared — elizabeth, mary
  male_names_historical_1920s ∩ male_names_modern_2024: 7 shared — charles, james, john, joseph, robert, thomas, william


In [5]:
import re
from collections import Counter
import pandas as pd

ERA_DIR = OUTPUT_DIR  # word_lists/era_names

# --- load the era name lists we just built ---
era_names = {}
for f in ERA_DIR.glob("*_names_*.txt"):
    sex, _, era = f.stem.split("_", 2)          # e.g. male_names_historical_1920s
    era_names.setdefault(era, {})[sex] = set(f.read_text(encoding="utf-8").split())

def tokenize(text):
    # MUST match the WEAT scoring cells — you used r'\b[a-z]+\b' on text.lower()
    return re.findall(r"\b[a-z]+\b", text.lower())

# --- coverage check over each (state, period) cell ---
rows = []
for (state, period), text in cell_corpus.items():   # <-- cell_corpus from your loader
    if period not in era_names:                      # only check the era that matches the cell
        continue
    toks   = tokenize(text)
    counts = Counter(toks)
    tok_set = set(toks)
    n_tokens = len(toks)
    for sex in ("male", "female"):
        names   = era_names[period][sex]
        present = names & tok_set
        hits    = sum(counts[n] for n in present)
        rows.append({
            "state": state, "period": period, "sex": sex,
            "names_total":   len(names),
            "names_present": len(present),
            "coverage_pct":  round(100 * len(present) / len(names), 1),
            "name_hits":     hits,
            "corpus_tokens": n_tokens,
            "hits_per_10k":  round(10000 * hits / n_tokens, 2) if n_tokens else 0,
        })

cov = pd.DataFrame(rows)
print(cov.groupby(["period", "sex"])[["names_present", "coverage_pct", "hits_per_10k"]].mean().round(2))
print("\nLowest-coverage cells:")
print(cov.sort_values("coverage_pct").head(10).to_string(index=False))

VAL_DIR = BASE_DIR / "outputs" / "validation"
VAL_DIR.mkdir(parents=True, exist_ok=True)
cov.to_csv(VAL_DIR / "era_name_coverage.csv", index=False)

NameError: name 'cell_corpus' is not defined